# Fraud model: time-based evaluation vs baselines

In [1]:
import json
ev = json.load(open('data/processed/evaluation.json'))
print(ev['experiment']['split']); print('rows:', ev['experiment']['train_rows'], ev['experiment']['val_rows'], ev['experiment']['test_rows'])

time-based: train first 60% / validation next 20% / test last 20%
rows: 12000 4000 4000


In [2]:
t = ev['fraud']['test']
print({k: t[k] for k in ['roc_auc','pr_auc','precision','recall','fpr','recall_at_1pct_fpr','brier','ece','chance_pr_auc']})

{'roc_auc': 0.9477, 'pr_auc': 0.7145, 'precision': 0.6418, 'recall': 0.6679, 'fpr': 0.0271, 'recall_at_1pct_fpr': 0.5498, 'brier': 0.0325, 'ece': 0.0175, 'chance_pr_auc': 0.0678}


In [3]:
import pandas as pd
rows = {'FlowFreeze': ev['fraud']['test'], **ev['fraud']['baselines']}
pd.DataFrame({k: {m: v.get(m) for m in ['precision','recall','pr_auc','roc_auc','fpr']} for k, v in rows.items()}).T

                               precision  recall  pr_auc  roc_auc     fpr
FlowFreeze                        0.6418  0.6679  0.7145   0.9477  0.0271
fixed_rules                       0.4679  0.1882     NaN      NaN  0.0156
logistic_regression               0.6030  0.4428  0.5481   0.9261  0.0212
isolation_forest_unsupervised     0.2647  0.1661  0.1801   0.7167  0.0335

In [4]:
import pandas as pd
pd.DataFrame(ev['slices']).T[['support','positives','recall','fpr']]

                           support  positives  recall     fpr
new_accounts_<90d            209.0       92.0  0.8261  0.0769
old_accounts_>=90d          3791.0      179.0  0.5866  0.0255
channel_app                 2419.0      122.0  0.6885  0.0226
channel_ussd                1068.0       46.0  0.4348  0.0176
channel_agent                513.0      103.0  0.7476  0.0756
to_merchant                  967.0        1.0  0.0000  0.0000
to_non_merchant             3033.0      270.0  0.6704  0.0366
legit_look_alike(unusual)    814.0        3.0  0.0000  0.0555
legit_merchant_traffic       967.0        1.0  0.0000  0.0000
legit_normal                1934.0        6.0  0.0000  0.0192

In [5]:
print(ev['fairness'])

{'max_fpr_ratio_across_groups': 4.37, 'flag': 'REVIEW: false-positive rate differs >2x between groups', 'most_harmed_legit_group': 'legit_look_alike(unusual)', 'note': 'FPR = share of legitimate transactions that would be alerted.'}


In [6]:
import json
im = json.load(open('data/processed/impact.json'))
pd.DataFrame(im['strategies']).T[['episodes_caught','preserved_pct','alerts','false_alerts','legit_value_held_bdt']]

                                episodes_caught  preserved_pct  alerts  false_alerts  legit_value_held_bdt
no_action                                   NaN            0.0     NaN           NaN                   0.0
rules + whole-wallet hold                  41.0           49.6   109.0          52.0             3358223.0
FlowFreeze + whole-wallet hold             67.0           55.6   282.0          82.0            11847550.0
FlowFreeze + proportional hold             67.0           55.6   282.0          82.0              516804.0